In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
import ROOT
import numpy as np

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x88711d0
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x895d140


In [3]:
# now get values for signal from the file
from fitHelper import load_fit_input
fit_input = load_fit_input("fit-configs/signal-only-mW-pol-new2-polfix/fit-inputs-0.0-0.0-mc_oo.txt")

In [4]:
pars = fit_input[0]
print(pars)
n_per_ab = fit_input[1]
means = np.asarray(fit_input[2])
C = np.asarray(fit_input[4]).reshape((len(pars), len(pars)))
print(means)
print(C)

ab = 5
n = n_per_ab * ab

['g1z', 'ka', 'la', 'mW', 'epol', 'ppol']
[-0.10543786 -0.07026024  0.09070786  0.09669952 -0.96116799  0.9610409 ]
[[ 1.48756379e+00 -2.53447028e-02 -8.45546719e-01  1.14781339e-02
   5.72692141e-02 -5.73024326e-02]
 [-2.53447028e-02  4.43502201e-01 -6.97917313e-02  1.26112387e-02
  -2.86545851e-02  2.84195449e-02]
 [-8.45546719e-01 -6.97917313e-02  2.41685276e+00  2.23332729e-03
  -3.22796549e-03  3.20841176e-03]
 [ 1.14781339e-02  1.26112387e-02  2.23332729e-03  9.18866394e-01
  -5.31969191e-03  5.34817067e-03]
 [ 5.72692141e-02 -2.86545851e-02 -3.22796549e-03 -5.31969191e-03
   3.07420656e-02 -9.22988133e-03]
 [-5.73024326e-02  2.84195449e-02  3.20841176e-03  5.34817067e-03
  -9.22988133e-03  3.09117561e-02]]


In [5]:
C_tilde = (C + np.outer(means, means)) * n

In [6]:
# V from C:
V_no_rate = np.linalg.inv(C) / n

# V from C_tilde:
V_rate = np.linalg.inv(C_tilde)

In [ ]:
precision = lambda M: np.sqrt(np.diag(M))

print("Precision without rate:")
p_no_rate = precision(V_no_rate)
print(p_no_rate)
print("Precision with rate:")
p_rate = precision(V_rate)
print(p_rate)

Precision without rate:
[0.00031079 0.00049922 0.00022994 0.00032962 0.0019744  0.00196721]
Precision with rate:
[0.00029125 0.00047446 0.00022774 0.00032948 0.00152739 0.0015275 ]


In [10]:
ratio = p_rate / p_no_rate
print("Precision ratio:")
for par, r in zip(pars, ratio):
    print(f"{par}: {r:.2f}")

# inverse of precision ratio is the rate improvement factor
improvement_factor = 1 / ratio
print("Rate improvement factor:")
for par, f in zip(pars, improvement_factor):
    print(f"{par}: {f:.2f}")

Precision ratio:
g1z: 0.94
ka: 0.95
la: 0.99
mW: 1.00
epol: 0.77
ppol: 0.78
Rate improvement factor:
g1z: 1.07
ka: 1.05
la: 1.01
mW: 1.00
epol: 1.29
ppol: 1.29
